# Neural Networks 1 - PyTorch Tutorial
---

## 목차 (Table of Contents)
1. Simple Mathematical Model - 퍼셉트론
2. Perceptron으로 논리 연산 구현 (AND, OR, NOT)
3. Perceptron의 한계 - XOR 문제

## 0. 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

---
## 1. Simple Mathematical Model - 퍼셉트론

### 뉴런의 수학적 모델

**Step 1: 가중합 (Weighted Summation)**
$$s = x_1 w_1 + x_2 w_2 + \cdots + x_n w_n = \sum_{i=1}^{n} x_i w_i$$

**Step 2: 활성화 함수 (Activation Function) - Hard Limit**
$$y = f(s) = \begin{cases} 1 & \text{if } s > 0 \\ 0 & \text{otherwise} \end{cases}$$

### 기하학적 의미
- 퍼셉트론은 입력 공간을 직선(초평면)으로 분리
- $x_1 w_1 + x_2 w_2 + w_3 = 0$ → 이 직선 위쪽이면 1, 아래쪽이면 0
- **선형 분리 가능한 문제만 풀 수 있다!**

In [ ]:
class Perceptron:
    def __init__(self, weights, bias):
        self.weights = torch.tensor(weights, dtype=torch.float32)
        self.bias = torch.tensor(bias, dtype=torch.float32)

    def forward(self, x):
        x = torch.tensor(x, dtype=torch.float32) if not isinstance(x, torch.Tensor) else x
        s = ????
        y = ????  # Hard limit (threshold)
        return y, s

    def __repr__(self):
        return f"Perceptron(weights={self.weights.tolist()}, bias={self.bias.item():.1f})"

### 활성화 함수 시각화

In [ ]:
s = torch.linspace(-5, 5, 200)

# Hard Limit (Step Function)
hard_limit = (s > 0).float()

# Sigmoid
sigmoid = torch.sigmoid(s)

fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

funcs = [
    (hard_limit, 'Hard Limit', 'navy'),
    (sigmoid, 'Sigmoid', 'crimson')
]

for ax, (y_vals, title, color) in zip(axes, funcs):
    ax.plot(s.numpy(), y_vals.numpy(), color=color, lw=2.5)
    ax.axhline(y=0, color='gray', lw=0.5, ls='--')
    ax.axvline(x=0, color='gray', lw=0.5, ls='--')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('s')
    ax.set_ylabel('f(s)')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. 퍼셉트론으로 논리 연산 구현

### 2.1 AND Gate
| x₁ | x₂ | y |
|:--:|:--:|:-:|
| 0  | 0  | 0 |
| 0  | 1  | 0 |
| 1  | 0  | 0 |
| 1  | 1  | 1 |

**문제(가중치)**: w₁=?, w₂=?, bias(w₃)=?


In [ ]:
# AND Gate
and_gate = Perceptron(weights=[??, ??], bias=??)
print(f"AND Gate: {and_gate}")
print("="*45)
print(f"{'x1':>4} {'x2':>4} {'Sum':>8} {'Output':>8}")
print("-"*45)

inputs = [[0,0], [0,1], [1,0], [1,1]]
for x in inputs:
    y, s = and_gate.forward(x)
    print(f"{x[0]:>4} {x[1]:>4} {s.item():>8.1f} {int(y.item()):>8}")

### 2.2 OR Gate
**문제(가중치)**: w₁=?, w₂=?, bias(w₃)=?

In [ ]:
# OR Gate
or_gate = Perceptron(weights=[??, ??], bias=??)
print(f"OR Gate: {or_gate}")
print("="*45)
print(f"{'x1':>4} {'x2':>4} {'Sum':>8} {'Output':>8}")
print("-"*45)

for x in inputs:
    y, s = or_gate.forward(x)
    print(f"{x[0]:>4} {x[1]:>4} {s.item():>8.1f} {int(y.item()):>8}")

### 2.3 NOT Gate
**문제(가중치)**: w₁=?, bias(w₂)=?

In [ ]:
# NOT Gate
not_gate = Perceptron(weights=[??], bias=??)
print(f"NOT Gate: {not_gate}")
print("="*35)
print(f"{'x1':>4} {'Sum':>8} {'Output':>8}")
print("-"*35)

for x in [[0], [1]]:
    y, s = not_gate.forward(x)
    print(f"{x[0]:>4} {s.item():>8.1f} {int(y.item()):>8}")

### 2.4 시각화
퍼셉트론은 직선(Decision Boundary)으로 입력 공간을 분리합니다.

In [ ]:
def plot_decision_boundary(weights, bias, title, ax, inputs, targets):
    """Visualize decision boundary of a perceptron"""
    # Decision boundary: w1*x1 + w2*x2 + bias = 0 -> x2 = -(w1*x1 + bias) / w2
    x1_range = np.linspace(-0.5, 1.5, 300)

    if abs(weights[1]) > 1e-6:
        x2_boundary = -(weights[0] * x1_range + bias) / weights[1]
        ax.plot(x1_range, x2_boundary, 'k--', lw=2, label='Decision Boundary')
        ax.fill_between(x1_range, x2_boundary, 2, alpha=0.15, color='blue', label='Output = 1')
        ax.fill_between(x1_range, x2_boundary, -1, alpha=0.15, color='red', label='Output = 0')

    # Data points
    for inp, target in zip(inputs, targets):
        color = 'blue' if target == 1 else 'red'
        marker = 'o' if target == 1 else 's'
        ax.scatter(inp[0], inp[1], c=color, s=200, marker=marker,
                   edgecolors='black', linewidth=2, zorder=5)
        ax.annotate(f'({inp[0]},{inp[1]})->{target}',
                    (inp[0]+0.05, inp[1]+0.08), fontsize=9)

    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x1', fontsize=12)
    ax.set_ylabel('x2', fontsize=12)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

inputs_2d = [[0,0], [0,1], [1,0], [1,1]]

# AND
plot_decision_boundary([1.0, 1.0], -1.5, 'AND Gate\nw1=1, w2=1, b=-1.5',
                       axes[0], inputs_2d, [0, 0, 0, 1])

# OR
plot_decision_boundary([1.0, 1.0], -0.5, 'OR Gate\nw1=1, w2=1, b=-0.5',
                       axes[1], inputs_2d, [0, 1, 1, 1])

# XOR (Not separable!)
ax = axes[2]
xor_targets = [0, 1, 1, 0]
for inp, target in zip(inputs_2d, xor_targets):
    color = 'blue' if target == 1 else 'red'
    marker = 'o' if target == 1 else 's'
    ax.scatter(inp[0], inp[1], c=color, s=200, marker=marker,
               edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'({inp[0]},{inp[1]})->{target}',
                (inp[0]+0.05, inp[1]+0.08), fontsize=9)

ax.text(0.5, 0.7, 'Cannot separate\nwith a single line!', fontsize=11,
        ha='center', color='red', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
ax.set_xlim(-0.5, 1.5)
ax.set_ylim(-0.5, 1.5)
ax.set_xlabel('x1', fontsize=12)
ax.set_ylabel('x2', fontsize=12)
ax.set_title('XOR Gate\n(Not Linearly Separable!)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 연습 문제

1. **AND, OR Gate의 가중치를 바꿔보세요**
(w₁=2.0, w₂=2.0, bias=-3.0 일 때도 AND가 동작하나요?)

---
## 3. 퍼셉트론의 한계 - XOR 문제

### XOR
| x₁ | x₂ | y |
|:--:|:--:|:-:|
| 0  | 0  | 0 |
| 0  | 1  | 1 |
| 1  | 0  | 1 |
| 1  | 1  | 0 |

XOR은 **하나의 직선으로 분리할 수 없습니다!**

### 해결: 다층 퍼셉트론 (MLP)
- 강의에서의 핵심 아이디어: **XOR = (NOT AND) AND OR**
  - 첫 번째 뉴런(y₁): AND 연산
  - 두 번째 뉴런(y₂): OR 연산
  - 출력 뉴런: y₂ AND (NOT y₁) → w₃₁=-1, w₃₂=1, bias=-0.5

In [ ]:
# Hidden layer neurons
neuron_and = ????   # y1: AND
neuron_or  = ????   # y2: OR

# Output layer neuron
neuron_out = ????  # y: NOT y1 AND y2

print(f"Hidden Neuron 1 (AND): {neuron_and}")
print(f"Hidden Neuron 2 (OR):  {neuron_or}")
print(f"Output Neuron:         {neuron_out}")
print()
print(f"{'x1':>4} {'x2':>4} | {'y1(AND)':>8} {'y2(OR)':>8} | {'y(XOR)':>8}")
print("-" * 50)

for x in inputs:
    y1, _ = neuron_and.forward(x)
    y2, _ = neuron_or.forward(x)
    y_out, _ = neuron_out.forward([y1.item(), y2.item()])
    print(f"{x[0]:>4} {x[1]:>4} | {int(y1.item()):>8} {int(y2.item()):>8} | {int(y_out.item()):>8}")

print()
print("XOR problem solved successfully!")

### XOR 네트워크의 공간 변환 시각화
Hidden layer가 입력 공간을 **새로운 공간으로 변환**하여 선형 분리 가능하게 만듭니다.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Original input space
ax = axes[0]
xor_colors = ['red', 'blue', 'blue', 'red']
xor_labels = [0, 1, 1, 0]
for inp, color, label in zip(inputs_2d, xor_colors, xor_labels):
    ax.scatter(inp[0], inp[1], c=color, s=300, edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'y={label}', (inp[0]+0.07, inp[1]+0.07), fontsize=11, fontweight='bold')
ax.set_title('(1) Original Input Space (x1, x2)\nNot Linearly Separable!', fontsize=12, fontweight='bold')
ax.set_xlabel('x1', fontsize=12)
ax.set_ylabel('x2', fontsize=12)
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Hidden layer output space
ax = axes[1]
hidden_outputs = []
for x in inputs_2d:
    y1, _ = neuron_and.forward(x)
    y2, _ = neuron_or.forward(x)
    hidden_outputs.append([y1.item(), y2.item()])

for h, color, label in zip(hidden_outputs, xor_colors, xor_labels):
    ax.scatter(h[0], h[1], c=color, s=300, edgecolors='black', linewidth=2, zorder=5)
    ax.annotate(f'y={label}', (h[0]+0.05, h[1]+0.05), fontsize=11, fontweight='bold')

# Decision boundary: -y1 + y2 - 0.5 = 0 -> y2 = y1 + 0.5
y1_range = np.linspace(-0.3, 1.3, 100)
y2_boundary = y1_range + 0.5
ax.plot(y1_range, y2_boundary, 'k--', lw=2, label='Decision Boundary')

ax.set_title('(2) Hidden Layer Output (y1, y2)\nLinearly Separable!', fontsize=12, fontweight='bold')
ax.set_xlabel('y1 (AND)', fontsize=12)
ax.set_ylabel('y2 (OR)', fontsize=12)
ax.set_xlim(-0.3, 1.3)
ax.set_ylim(-0.3, 1.3)
ax.grid(True, alpha=0.3)
ax.set_aspect('equal')

# Space transformation explanation
ax = axes[2]
ax.text(0.5, 0.85, 'Space Transformation', fontsize=14, ha='center', fontweight='bold',
        transform=ax.transAxes)
ax.text(0.5, 0.7, '(0,0)->(0,0)  y=0->y=0', fontsize=11, ha='center',
        transform=ax.transAxes, family='monospace')
ax.text(0.5, 0.55, '(0,1)->(0,1)  y=1->y=1', fontsize=11, ha='center',
        transform=ax.transAxes, family='monospace', color='blue')
ax.text(0.5, 0.4, '(1,0)->(0,1)  y=1->y=1', fontsize=11, ha='center',
        transform=ax.transAxes, family='monospace', color='blue')
ax.text(0.5, 0.25, '(1,1)->(1,1)  y=0->y=0', fontsize=11, ha='center',
        transform=ax.transAxes, family='monospace')
ax.text(0.5, 0.08, '(0,1) and (1,0) map to the same point!\n-> Now linearly separable!', fontsize=11,
        ha='center', transform=ax.transAxes, color='green', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='lightyellow'))
ax.axis('off')

plt.tight_layout()
plt.show()

---
## 연습 문제

2. **NAND 게이트를 만들어 보세요**: NAND 연산을 구하는 가중치(w1, w1, bias)를 찾으시오.
3. **NAND 게이트만으로 XOR 구현하기**